In [ ]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/s1156052/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit > /dev/null


In [ ]:
# Collect once (dated snapshot on Drive): every CEDAR neoantigen with a disease_group, plus viral/TAA background negatives
import pandas as pd
from src import config, pancancer
pd.set_option("display.width", 200)

raw = pancancer.get_raw(config.DATA_DIR, refresh=False)
print(raw.groupby("group").size())


In [ ]:
# How much data each disease has (decides which diseases can be their own train/test arm)
pancancer.summary(raw)


In [ ]:
# Disease names that fell into "other": check none of them belongs to a named group or is not a cancer
pancancer.discover_terms(raw).head(40)


In [ ]:
# Step 2: frozen dataset = one manifest (unique peptides, disease_group, CD-HIT cluster, outer fold). Needs cd-hit (Colab).
# neg_mode "exp": disease-matched T cell-negative neoantigens (primary) | "bg": viral/TAA background (the paper's choice)
from src import pancancer_split as ps
NAME, NEG_MODE = "pancancer_cedar_v1", "exp"
manifest, summary = ps.make_dataset(raw, config.DATA_DIR, config.WORK_DIR, name=NAME, neg_mode=NEG_MODE)
summary


In [ ]:
# One arm (train diseases) x one outer test fold -> the dataframe the runner expects; written to DATA_DIR/<NAME>/<arm>_f<k>/
ARM, FOLD = "pooled", 0
df = ps.arm_frame(manifest, ARM, FOLD, neg_ratio=5)
ps.write_arm(df, config.DATA_DIR, NAME)
ps.eval_table(df)     # N+/N- per disease on test: all rows vs rows not similar to training (reportable needs >= 50 each)


In [ ]:
# Every arm x fold (cheap; each frame is checked for sequence/cluster leakage)
for arm in ps.ARMS:
    for k in range(5):
        ps.write_arm(ps.arm_frame(manifest, arm, k), config.DATA_DIR, NAME)
